In [ ]:
import numpy as np
from sim_library.constants import k_eff, dR, m, kb, omega_eg
from scipy.constants import hbar, pi
import matplotlib.pyplot as plt
from pathlib import Path
from sim_library.simulation import simulate_pulses_single_atom, simulate_pulses_single_atom_analyt, simulate_pulses_p_dist, simulate_alg_cooling, simulate_alg_cooling_custom
from sim_library.sequences import RR3_gate, PulseSequence, UpPulse, DownPulse, FreeEvolution, gen_MDFEup_seq, gen_MDFEdown_seq, RR3_adjMDFE_500kHz
from sim_library.plotting import plot_state_trajectories, plot_bloch, plot_hist, save_gif, plot_coolingcycles_22, fit_gaussian, fit_gaussian_custom, display_table_compare, plot_bloch_multi
from sim_library.data_io import load_p_dists

In [ ]:

# Rabi frequency in 2*pi*Hz units
rabi_freq = 2*pi*5e5
rabi_time = 2*pi/rabi_freq # 2pi pulse time

# Set total resolution of time steps
n_steps = 1

indices = [8,10,12,14]

# Define basis: p in units of hbar*k (even p = ground state)
p_min = -8
p_max = 8
basis = np.arange(p_min, p_max + 1)

target_states = np.full((len(indices),2),1/np.sqrt(2), dtype=np.complex128)
target_states[:,1] = target_states[:,1]*np.exp(1j*pi*np.array([0.125,0.625,1.125,1.625]))

free_times = np.linspace(0.95*pi/dR, 1.05*pi/dR, 10000)

fidelities = np.zeros((len(indices),len(free_times)))
# fid_sum = np.zeros(len(free_times))

# Set doppler shift
atom_veloc = 0.0 # Stationary

for i in range(len(indices)):
    initial_state = np.zeros(shape=len(basis), dtype=np.complex128)
    initial_state[indices[i]] = 1
    initial_state[indices[i]+1] = 1
    initial_state = initial_state/ np.linalg.norm(initial_state) # Normalise

    for t, time in enumerate(free_times):
        pulse_seqq = PulseSequence()
        detuning = omega_eg/2
        free_time = 2*pi/(np.abs(dR-detuning))
        free = FreeEvolution(laser_det=(np.full(n_steps, -detuning)), duration=(free_time/2))
        MDFE = gen_MDFEdown_seq(free_time=time, rabi_freq=rabi_freq, time_steps=n_steps, detuning=dR)
        pulse_seqq.add_pulses([MDFE, free])
        wave_func, _ = simulate_pulses_single_atom_analyt(pulse_seq=pulse_seqq, basis=basis, initial_state=initial_state, d_shift=k_eff*atom_veloc)

        fidelity = np.abs(np.vdot(wave_func[-1, indices[i]:indices[i]+2],target_states[i,:]))**2
        fidelities[i, t] = fidelity
        

### Individual fidelities ###
fig, ax = plt.subplots()
ax.plot(free_times*dR/pi,fidelities.transpose())

peak_locs = free_times[np.argmax(fidelities, axis=1)]
print(peak_locs*dR/pi)
print(np.max(fidelities, axis=1))

# for i in range(len(indices)):
#     ax.plot(np.full(2,peak_locs[i]*dR/pi), [0,1])

ax.set_ylim(0,1.05)
ax.set_xlim(free_times[0]*dR/pi, free_times[-1]*dR/pi)
ax.set_xlabel(r'$t$ $(\pi/\delta_R)$')
ax.set_ylabel('Fidelity')
ax.set_title(r'$G_+(\pi/8)$ time scan at 500kHz')
ax.legend([r'$n=0,1$',r'$n=2,3$',r'$n=4,5$',r'$n=6,7$'], loc='lower right')
ax.grid()
######

### Average fidelity ###
fid_sum = np.sum(fidelities, axis=0)/len(indices)
print(free_times[np.argmax(fid_sum)]*dR/pi)
print(np.max(fid_sum))
fig, ax = plt.subplots()
ax.plot(free_times*dR/pi,fid_sum)



ax.set_ylim(0,1.05)
ax.set_xlim(free_times[0]*dR/pi, free_times[-1]*dR/pi)
ax.set_xlabel(r'$t$ $(\pi/\delta_R)$')
ax.set_ylabel('Averaged Fidelity')
ax.set_title(r'$G_+(\pi/8)$ time scan at 500kHz')
ax.grid()
######




In [ ]:
exchange10 = PulseSequence()

    pulse_1 = DownPulse(laser_det=np.full(time_steps, detuning), phase=np.full(time_steps, 0), rabi_freq=np.full(time_steps, rabi_freq), duration=rabi_time/4) #down pi/2 pulse, phase = pi/2

    MDFE_1 = gen_MDFEup_seq(free_time=pi/(4*dR), rabi_freq=rabi_freq, time_steps=time_steps, detuning=detuning) # pi/4
    freevolve_1 = FreeEvolution(laser_det=(np.full(time_steps, +detuningfree)), duration=5*free_time/8) #5pi/4 
    freevolve_2 = FreeEvolution(laser_det=(np.full(time_steps, +detuningfree)), duration=free_time/2) #pi

    exchange10.add_pulses([pulse_1,
                    freevolve_1,
                    MDFE_1,
                    pulse_1,
                    freevolve_2])